# 02 — Preprocessing and Targets

All preprocessing decisions in one place, behind reusable functions in
`ujiindoorloc.preprocessing`. The same code paths are used by every
later notebook.

In [1]:
# --- bootstrap: make src/ importable when notebook started outside `uv run` ---
import sys
from pathlib import Path

_HERE = Path.cwd()
for parent in [_HERE, *_HERE.parents]:
    if (parent / "src" / "ujiindoorloc").is_dir():
        if str(parent / "src") not in sys.path:
            sys.path.insert(0, str(parent / "src"))
        break

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [2]:
import numpy as np
import pandas as pd
import joblib

from ujiindoorloc.constants import (
    COMBINED_TARGET, LEAKAGE_COLUMNS, MISSING_SIGNAL_VALUE,
    REPLACEMENT_SIGNAL_VALUE, MODELS_DIR,
)
from ujiindoorloc.data_loading import load_raw_data, split_features_targets, get_wap_columns
from ujiindoorloc.preprocessing import prepare_classification_data
from ujiindoorloc.reporting import ensure_report_dirs, save_table

ensure_report_dirs()
raw = load_raw_data()
print("MISSING_SIGNAL_VALUE :", MISSING_SIGNAL_VALUE)
print("REPLACEMENT_SIGNAL   :", REPLACEMENT_SIGNAL_VALUE)

MISSING_SIGNAL_VALUE : 100
REPLACEMENT_SIGNAL   : -110


## 1. Target construction

`building_floor` is built from `BUILDINGID` and `FLOOR`. We align the
categorical class set across train+valid so confusion-matrix axes are
stable.

In [3]:
split = split_features_targets(raw.train, raw.valid, target_name=COMBINED_TARGET)
print("X_train:", split.X_train.shape)
print("X_valid:", split.X_valid.shape)
print("classes:", list(split.y_train.cat.categories))

X_train: (19937, 520)
X_valid: (1111, 520)
classes: ['B0_F0', 'B0_F1', 'B0_F2', 'B0_F3', 'B1_F0', 'B1_F1', 'B1_F2', 'B1_F3', 'B2_F0', 'B2_F1', 'B2_F2', 'B2_F3', 'B2_F4']


## 2. Preprocessing pipelines

Two variants, both fit **on training only**:

* **scaled** — replace 100→-110, drop train-constant cols, StandardScaler.
  Used by LR / LDA / QDA / kNN / PCA.
* **tree** — replace 100→-110, drop train-constant cols, *no scaling*.
  Used by Decision Tree / Random Forest.

In [4]:
prepared = prepare_classification_data(split.X_train, split.X_valid)
print("X_train_scaled :", prepared.X_train_scaled.shape)
print("X_valid_scaled :", prepared.X_valid_scaled.shape)
print("X_train_tree   :", prepared.X_train_tree.shape)
print("X_valid_tree   :", prepared.X_valid_tree.shape)
print("kept WAPs (after train-constant filter):", len(prepared.kept_wap_columns))

X_train_scaled : (19937, 465)
X_valid_scaled : (1111, 465)
X_train_tree   : (19937, 465)
X_valid_tree   : (1111, 465)
kept WAPs (after train-constant filter): 465


## 3. Save the manifest of kept WAP columns

In [5]:
kept_df = pd.DataFrame({"wap": prepared.kept_wap_columns})
save_table(kept_df, "selected_wap_columns_after_constant_filter.csv")
kept_df.head()

,wap
0,WAP001
1,WAP002
2,WAP005
3,WAP006
4,WAP007


## 4. Quality checks

In [6]:
assert prepared.X_train_scaled.shape[0] == len(split.y_train)
assert prepared.X_valid_scaled.shape[0] == len(split.y_valid)
assert prepared.X_train_scaled.shape[1] == prepared.X_valid_scaled.shape[1]
assert prepared.X_train_tree.shape[1]   == prepared.X_valid_tree.shape[1]
assert not np.isnan(prepared.X_train_scaled).any()
assert not np.isnan(prepared.X_valid_scaled).any()

# Leakage guard — no metadata/target column may be among the predictors.
bad = set(split.X_train.columns) & set(LEAKAGE_COLUMNS)
assert not bad, f"leakage columns in features: {bad}"

# Same WAP order in train and validation (we already filter to wap_cols upstream).
assert list(split.X_train.columns) == list(split.X_valid.columns)

print("All preprocessing quality checks passed.")

All preprocessing quality checks passed.


## 5. Persist fitted pipelines

Saved so any later notebook can re-use the *exact same* transformation
without re-fitting.

In [7]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(prepared.scaled_pipeline, MODELS_DIR / "preprocessor_scaled.joblib")
joblib.dump(prepared.tree_pipeline,   MODELS_DIR / "preprocessor_tree.joblib")
print("Saved preprocessors to:", MODELS_DIR)

Saved preprocessors to: /Users/andrejvysny/fiit/oznal-python/models


## 6. Why fit on training only?

Fitting any preprocessing step on validation data leaks information from
the future *test set* back into model selection. Concretely:

* **Constant-column filter** — a column that is constant in training but
  varies in validation gives the model no signal during training; we keep
  it filtered out and let the model live with that.
* **StandardScaler** — mean/std must come from training only, otherwise
  the validation distribution co-determines the model input.
* **PCA / LDA** (Scenario 5) — same principle: `fit_transform(train)`,
  then `transform(valid)`.